Задание 4. Seed-and-Extend

Seed-and-Extend — двухфазный эвристический алгоритм для поиска локального выравнивания, который заменяет полное вычисление матрицы Смита–Уотермана за O(n×m) на приближённое решение, работающее значительно быстрее на практике.

In [20]:
!pip install biopython matplotlib numpy -q

import numpy as np
from Bio import SeqIO, Entrez
import matplotlib.pyplot as plt
import urllib.request
import os


In [30]:

ref = "CTAGGATCCAGGCTACTACA"
query = "GGATCCATCTATA"
k = 4
X = 2
match_score = 1
mismatch_score = -1

# Индекс k-меров референса
index = {}
for i in range(len(ref)-k+1):
    kmer = ref[i:i+k]
    index.setdefault(kmer, []).append(i)

# Поиск всех seeds
seeds = []
for i in range(len(query)-k+1):
    kmer = query[i:i+k]
    if kmer in index:
        for j in index[kmer]:
            seeds.append((i, j, kmer))

print("Индекс k-меров (первые 5):")
for km, pos in list(index.items())[:5]:
    print(f"  {km}: {pos}")

print(f"\nВсего seeds: {len(seeds)}")
for s in seeds:
    print(f"  Seed query[{s[0]}:{s[0]+k}] = {s[2]}, ref[{s[1]}:{s[1]+k}]")

# Функция расширения в одну сторону с X-drop
def extend_one_side(seq1, seq2, start1, start2, k, direction, match=1, mismatch=-1, X=2):

    step = -1 if direction == -1 else 1
    if step == -1:
        i = start1 - 1
        j = start2 - 1
    else:
        i = start1 + k
        j = start2 + k

    cur_score = k * match   # Scur
    max_score = cur_score   # Smax
    max_i, max_j = start1, start2
    scur_list = [cur_score]

    while 0 <= i < len(seq1) and 0 <= j < len(seq2):
        pair_score = match if seq1[i] == seq2[j] else mismatch
        cur_score += pair_score
        scur_list.append(cur_score)
        if cur_score > max_score:
            max_score = cur_score
            max_i, max_j = i, j
        if max_score - cur_score >= X:
            break
        i += step
        j += step

    # Формируем расширенные подстроки
    if direction == -1:   # влево
        left_part1 = seq1[max_i:start1]
        left_part2 = seq2[max_j:start2]
        full1 = left_part1 + seq1[start1:start1+k]
        full2 = left_part2 + seq2[start2:start2+k]
    else:                 # вправо
        right_part1 = seq1[start1+k:max_i+1]
        right_part2 = seq2[start2+k:max_j+1]
        full1 = seq1[start1:start1+k] + right_part1
        full2 = seq2[start2:start2+k] + right_part2
    return full1, full2, max_score, scur_list

# Обработка всех seeds, выбор лучшего
best_score = -999
best_alignment = None
best_details = None

for (i, j, kmer) in seeds:
    left1, left2, smax_left, scur_left = extend_one_side(query, ref, i, j, k, direction=-1)
    right1, right2, smax_right, scur_right = extend_one_side(query, ref, i, j, k, direction=1)


    combined1 = left1[:-k] + right1
    combined2 = left2[:-k] + right2
    total_smax = smax_left + smax_right - k * match_score

    if total_smax > best_score:
        best_score = total_smax
        best_alignment = (combined1, combined2)
        best_details = (i, j, kmer, smax_left, scur_left, smax_right, scur_right)



print(f"Лучший seed: {best_details[2]} (позиции: query={best_details[0]}, ref={best_details[1]})")
print(f"Smax (влево): {best_details[3]}")
print(f"Scur (влево): {best_details[4]}")
print(f"Smax (вправо): {best_details[5]}")
print(f"Scur (вправо): {best_details[6]}")
print(f"Итоговый Smax: {best_score}")
print("Выравнивание:")
print(best_alignment[0])
print(best_alignment[1])

Индекс k-меров (первые 5):
  CTAG: [0]
  TAGG: [1]
  AGGA: [2]
  GGAT: [3]
  GATC: [4]

Всего seeds: 4
  Seed query[0:4] = GGAT, ref[3:7]
  Seed query[1:5] = GATC, ref[4:8]
  Seed query[2:6] = ATCC, ref[5:9]
  Seed query[3:7] = TCCA, ref[6:10]
Лучший seed: GGAT (позиции: query=0, ref=3)
Smax (влево): 4
Scur (влево): [4]
Smax (вправо): 7
Scur (вправо): [4, 5, 6, 7, 6, 5]
Итоговый Smax: 7
Выравнивание:
GGATCCA
GGATCCA
